# 10 - Agent Examples

Demonstrates the **orchestrator + worker tools** pattern using three ready-to-use example agents from `aimu.agents.examples`.

Each agent class wraps an orchestrator `SimpleAgent` that coordinates specialist worker sub-agents exposed as MCP tools. The LLM decides which workers to call, in what order, and how many times.

| Agent | Workers |
|---|---|
| `ResearchReportAgent` | `research_overview`, `find_examples`, `find_counterpoints` |
| `CodeReviewAgent` | `review_security`, `review_performance`, `review_readability` |
| `ContentCreationAgent` | `research_topic`, `create_outline`, `write_section` |

---
## ResearchReportAgent

An orchestrator `SimpleAgent` autonomously coordinates three worker sub-agents. The LLM decides which workers to call, in what order, and how many times — rather than following a predetermined code path.

```
research-report-agent  (orchestrator)
  ├─ [tool] research_overview   →  overview-worker      (SimpleAgent)
  ├─ [tool] find_examples       →  examples-worker      (SimpleAgent)
  └─ [tool] find_counterpoints  →  counterpoints-worker (SimpleAgent)
```

### Setup

Create an `OllamaClient` and pass it to `ResearchReportAgent`. The agent internally creates fresh client instances for each worker.

In [ ]:
from aimu.models.ollama import OllamaClient
from aimu.agents.examples import ResearchReportAgent

model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
agent = ResearchReportAgent(model_client)

print("Orchestrator model:", model_client.model.value)
print("Worker tools:", [t.name for t in model_client.mcp_client.list_tools()])

### Run

Call `agent.run()` with a research topic. The orchestrator will call each worker tool, then synthesize the results into a final report.

In [ ]:
task = "What is retrieval-augmented generation?"

result = agent.run(task)
print(result)

Inspect the orchestrator's message history to see the full tool-calling sequence.

In [ ]:
agent.messages

### Streaming

`run(stream=True)` yields `AgentChunk` objects. `TOOL_CALLING` chunks show which worker was invoked and the worker's response; `GENERATING` chunks are the orchestrator's synthesis.

In [ ]:
from aimu.models import StreamPhase

task = "What is retrieval-augmented generation?"

for chunk in agent.run(task, stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        worker = chunk.content["name"]
        response_preview = chunk.content["response"][:80].replace("\n", " ")
        print(f"\n[tool: {worker}]\n  {response_preview}...\n")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

In [ ]:
del agent, model_client

---
## CodeReviewAgent

An orchestrator `SimpleAgent` autonomously coordinates three specialist reviewer sub-agents. The LLM decides which reviewers to engage based on the submitted code.

```
code-review-agent  (orchestrator)
  ├─ [tool] review_security     →  security-reviewer    (SimpleAgent)
  ├─ [tool] review_performance  →  performance-reviewer (SimpleAgent)
  └─ [tool] review_readability  →  readability-reviewer (SimpleAgent)
```

### Setup

Create an `OllamaClient` and pass it to `CodeReviewAgent`.

In [ ]:
from aimu.models.ollama import OllamaClient
from aimu.agents.examples import CodeReviewAgent

model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
agent = CodeReviewAgent(model_client)

print("Orchestrator model:", model_client.model.value)
print("Reviewer tools:", [t.name for t in model_client.mcp_client.list_tools()])

### Sample Code

A short snippet with an SQL injection vulnerability and an O(n²) search loop — two issues that span different review dimensions.

In [ ]:
code_snippet = '''
import sqlite3

def get_user(db_path, username):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # Fetch user record
    cursor.execute("SELECT * FROM users WHERE name = '" + username + "'")
    return cursor.fetchone()

def find_duplicates(items):
    dupes = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j] and items[i] not in dupes:
                dupes.append(items[i])
    return dupes
'''

### Run

Pass the code to `agent.run()`. The orchestrator will call each specialist reviewer, then synthesize findings into a prioritized report.

In [ ]:
result = agent.run(f"Review this Python code:\n\n{code_snippet}")
print(result)

Inspect the orchestrator's message history to see each reviewer call and its response.

In [ ]:
agent.messages

### Streaming

`TOOL_CALLING` chunks reveal which reviewer was called and a preview of their findings; `GENERATING` chunks are the orchestrator's synthesis.

In [ ]:
from aimu.models import StreamPhase

for chunk in agent.run(f"Review this Python code:\n\n{code_snippet}", stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        reviewer = chunk.content["name"]
        response_preview = chunk.content["response"][:80].replace("\n", " ")
        print(f"\n[tool: {reviewer}]\n  {response_preview}...\n")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

In [ ]:
del agent, model_client

---
## ContentCreationAgent

An orchestrator `SimpleAgent` builds content step by step. The LLM determines how many sections to write and how many times to call each worker.

```
content-creation-agent  (orchestrator)
  ├─ [tool] research_topic   →  research-worker  (SimpleAgent)
  ├─ [tool] create_outline   →  outline-worker   (SimpleAgent)
  └─ [tool] write_section    →  section-writer   (SimpleAgent)
```

### Setup

Create an `OllamaClient` and pass it to `ContentCreationAgent`.

In [ ]:
from aimu.models.ollama import OllamaClient
from aimu.agents.examples import ContentCreationAgent

model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
agent = ContentCreationAgent(model_client)

print("Orchestrator model:", model_client.model.value)
print("Worker tools:", [t.name for t in model_client.mcp_client.list_tools()])

### Run

Give the agent a content brief. It will research the topic, build an outline, write each section, then assemble the final piece.

In [ ]:
task = "Write about the benefits of test-driven development as a blog post for software developers"

result = agent.run(task)
print(result)

Inspect the orchestrator's message history to trace the research → outline → write loop.

In [ ]:
agent.messages

### Streaming

`TOOL_CALLING` chunks show which worker was called (and reveal the step-by-step construction); `GENERATING` chunks are the orchestrator assembling the final output.

In [ ]:
from aimu.models import StreamPhase

task = "Write about the benefits of test-driven development as a blog post for software developers"

for chunk in agent.run(task, stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        worker = chunk.content["name"]
        response_preview = chunk.content["response"][:80].replace("\n", " ")
        print(f"\n[tool: {worker}]\n  {response_preview}...\n")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## Clean Up

In [ ]:
del agent, model_client